# Data Analysis

In [1]:
# Pour installer les librairies requises
%pip install -q -r ../../requirements.txt


# si on modifie un module, il est rechargé automatiquement
%load_ext autoreload
%autoreload 2

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
sys.path.append("../../modules")

from intern.ihf import utils as utils
from intern.ihf.ban import extract as extract
from intern.ihf.ban import plot as plot

from intern.ihf.bd_forests import extract as extract_bdf
from intern.ihf.bd_forests import plot as plot_bdf

In [3]:
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt

In [4]:
import importlib
importlib.reload(utils)
importlib.reload(extract)
importlib.reload(plot)

importlib.reload(extract_bdf)
importlib.reload(plot_bdf)

<module 'intern.ihf.bd_forests.plot' from '/home/onyxia/work/Groupe_KC_AHA_Python/main/ihf/../../modules/intern/ihf/bd_forests/plot.py'>

A noter que le code suivant peut prendre du temps car il télécharge les données sur les forêts hébergées sur Onyxia (du fait de leur lourdeur).

In [5]:
adr = extract.read_concatenated_ban_files(extract.BAN_ADRESS_FILES)
occ = extract_bdf.read_bd_forests_file()
occ = occ[occ["exo_19"] == "BD FORET"]

/home/onyxia/work/Groupe_KC_AHA_Python/main/ihf/../../modules/intern/ihf/ban/extract.py:40: DtypeWarning: Columns (16: nom_ld) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep=";") # sep=";" the ihf ban files are semicolon separated


Téléchargement de https://minio.lab.sspcloud.fr/khcoulibaly/diffusion/occupation_du_sol.zip


In [39]:
occ = occ.to_crs(utils.CRS_M)
adr = adr[adr.type_position.isin(["bâtiment", "entrée"])]   # adjust after value_counts
adr = adr[adr.numero < 99999]                                # drops lieux-dits (BAN convention)
pts = (gpd.GeoDataFrame(adr, geometry=gpd.points_from_xy(adr.lon, adr.lat), crs=4326)
         .to_crs(utils.CRS_M).reset_index(drop=True))

xmin, ymin, xmax, ymax = pts.total_bounds

In [40]:
print(xmin, ymin, xmax, ymax)

808919.216116847 6214615.463904612 1072263.8937493423 6448547.30818032


In [ ]:
forest_typed = (occ.cx[xmin-utils.D_MAX:xmax+utils.D_MAX, ymin-utils.D_MAX:ymax+utils.D_MAX]
                [["niv3_19", "lib3_19", "geometry"]])
forest_mask = forest_typed.dissolve().explode(index_parts=False).reset_index(drop=True)

In [ ]:
forest_typed.head()

In [ ]:
def nearest(points, polys, dmax=None, cols=("geometry",)):
    j = gpd.sjoin_nearest(points[["geometry"]], polys[list(cols)], how="left",
                          max_distance=dmax, distance_col="d")
    return j[~j.index.duplicated()]          # ties create duplicates

pts["d_forest"] = nearest(pts, forest_mask, utils.D_MAX)["d"]

# addresses inside forest: negative distance to the edge
inside = pts.d_forest == 0
edges = gpd.GeoDataFrame(geometry=forest_mask.boundary, crs=utils.CRS_M)
pts.loc[inside, "d_forest"] = -nearest(pts[inside], edges)["d"]

pts["forest_type_nearest"] = nearest(pts, forest_typed, utils.D_MAX,
                                     cols=("geometry", "lib3_19"))["lib3_19"]

In [ ]:
def forest_share(pts, forest_typed, r):
    buf = pts[["geometry"]].copy()
    buf["geometry"] = buf.buffer(r, resolution=8)
    buf["pid"] = buf.index
    inter = gpd.overlay(buf, forest_typed[["lib3_19", "geometry"]],
                        how="intersection", keep_geom_type=True)
    inter["a"] = inter.area
    share = (inter.groupby("pid")["a"].sum() / buf.area).reindex(pts.index).fillna(0)
    dom = (inter.groupby(["pid", "lib3_19"])["a"].sum().reset_index()
                .sort_values("a").drop_duplicates("pid", keep="last")
                .set_index("pid")["lib3_19"])
    return share, dom.reindex(pts.index)

for r in utils.BUFFERS:
    pts[f"forest_{r}m"], pts[f"dom_type_{r}m"] = forest_share(pts, forest_typed, r)

In [ ]:
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

In [ ]:
def cluster_size(xy, r):
    tree = cKDTree(xy)
    pairs = tree.query_pairs(2 * r, output_type="ndarray")   # buffers overlap iff distance <= 2r
    A = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(len(xy),) * 2)
    _, lab = connected_components(A, directed=False)
    return np.bincount(lab)[lab], lab

xy = np.c_[pts.geometry.x, pts.geometry.y]
pts["cluster_size"], pts["cluster"] = cluster_size(xy, utils.R_CLUSTER)

TYPO_BINS, LABELS = [0, 3, 30, np.inf], ["isolated", "grouped", "dense"]
pts["typology"] = pd.cut(pts.cluster_size, TYPO_BINS, labels=LABELS)
pts["band"] = pd.cut(pts.d_forest.fillna(np.inf), [-np.inf, 0, 50, 100, 200, np.inf],
                     labels=["in forest", "≤50 m", "50–100 m", "100–200 m", ">200 m"])
pts["ihf"] = pts.d_forest <= 100

In [ ]:
for r in (30, 50, 100):
    s, _ = cluster_size(xy, r)
    print(r, pd.cut(s, TYPO_BINS, labels=LABELS).value_counts(normalize=True).round(3).to_dict())
    

In [ ]:
palette = {"in forest": "#1b5e20", "≤50 m": "#d73027", "50–100 m": "#fc8d59",
           "100–200 m": "#fee08b", ">200 m": "#d9d9d9"}
ct = pd.crosstab(pts.typology, pts.band, normalize="index").mul(100)
ax = ct.plot.barh(stacked=True, color=[palette[c] for c in ct.columns], figsize=(9, 3.5), width=.7)
ax.set(xlabel="% of addresses", ylabel="", title="Distance to forest edge by settlement type")
ax.legend(title="Distance", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
ax.spines[["top", "right"]].set_visible(False)

pd.crosstab(pts.typology, pts.band, margins=True).style.background_gradient(cmap="YlOrRd", axis=None)

In [ ]:
key="cb1_5bwe_1_9ba5234936bb78fffbab1218"

In [ ]:
from shapely import box
CELL = 500
pts["gx"], pts["gy"] = (pts.geometry.x // CELL).astype(int), (pts.geometry.y // CELL).astype(int)
grid = pts.groupby(["gx", "gy"]).agg(n=("ihf", "size"), p_ihf=("ihf", "mean")).reset_index()
grid = gpd.GeoDataFrame(grid, geometry=[box(x*CELL, y*CELL, (x+1)*CELL, (y+1)*CELL)
                        for x, y in zip(grid.gx, grid.gy)], crs=utils.CRS_M)
CARTO = f"https://basemaps.cartocdn.com/rastertiles/voyager/{{z}}/{{x}}/{{y}}.png?key={key}"
ATTR = "© OpenStreetMap contributors © CARTO"
grid[grid.n >= 5].explore("p_ihf", cmap="YlOrRd", vmin=0, vmax=1, tiles=CARTO, attr=ATTR,
                          style_kwds=dict(weight=0, fillOpacity=.75),
                          legend_kwds=dict(caption="Share of addresses ≤100 m from forest"))

In [ ]:
import folium
sub = pts[pts.nom_commune == "Allons"]
b = sub.total_bounds
m = forest_typed.cx[b[0]-300:b[2]+300, b[1]-300:b[3]+300].explore(
        column="lib3_19", cmap="Greens", tiles=CARTO, attr=ATTR,
        style_kwds=dict(weight=.3), legend=False, name="Forest")
sub[["numero", "nom_voie", "d_forest", "typology", "band", "geometry"]].explore(
        m=m, column="band", categorical=True, cmap=list(palette.values()),
        marker_kwds=dict(radius=3), tooltip=["numero", "nom_voie", "d_forest", "typology"],
        name="Addresses")
folium.LayerControl().add_to(m); m

_______________________________________________